In [2]:
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler

# 1. LOAD & FIX: Handles titles with commas that break standard CSV readers
def robust_load(file_path):
    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        header = lines[0].strip().split(',')
        for line in lines[1:]:
            parts = line.strip().split(',')
            if len(parts) >= 17:
                # Keep first 3 cols and last 13 cols; join the rest as the title
                gse, gsm, gpl = parts[0], parts[1], parts[2]
                numeric_vals = parts[-13:]
                title = ",".join(parts[3:-13])
                rows.append([gse, gsm, gpl, title] + numeric_vals)
    df = pd.DataFrame(rows, columns=header)
    for col in header[4:]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

# 2. LABEL: Apply the SLE vs Healthy logic
def get_label(title):
    t = str(title).lower()
    if 'healthy' in t: return 'Healthy'
    if any(term in t for term in ['sle', 'sl_', 'glom', 'tub', 'lupus']): return 'SLE'
    if re.search(r'^dna\d', t) or re.search(r'^\d{6}_', t): return 'SLE'
    return 'Other'

# Execute Process
df = robust_load('database.csv')
df['label'] = df['title'].apply(get_label)

# 3. NORMALIZE: Apply Z-score to numeric columns (MTHFR to STAT4)
numeric_cols = df.columns[4:-1] # Excludes ID cols and the 'label' col
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Save the final result
df.to_csv('normalized_labeled_database.csv', index=False)
print("Data normalized and saved to 'normalized_labeled_database.csv'")

Data normalized and saved to 'normalized_labeled_database.csv'
